# Severstal OSR Full Experiment

This notebook runs the full `severstral-osr` pipeline:
1. export dataset stats and single-label lists
2. train stage-1 PatchCore once
3. run split pipelines (classifier + embeddings + OSR + cascade)
4. aggregate metrics

In [ ]:
# Optional compatibility reinstall (run only if your environment has package conflicts)
"""import sys, subprocess

# Reinstall a compatible scientific stack
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--upgrade", "--force-reinstall",
    "numpy==1.26.4",
    "scipy==1.11.4",
    "scikit-learn==1.4.2",
    "pandas==2.2.2",
    "matplotlib==3.8.4",
    "pillow==10.3.0",
    "pyyaml==6.0.1",
    "tqdm==4.66.4",
])

print("Done. Now restart runtime: Runtime > Restart runtime")"""

In [ ]:
# Setup paths (Colab-friendly)
import os, sys, subprocess
from pathlib import Path

REPO = Path('/content/FYP-code')
if not REPO.exists():
    subprocess.check_call(['git', 'clone', 'https://github.com/spinelessknave8/FYP_code.git', str(REPO)])
else:
    subprocess.check_call(['git', '-C', str(REPO), 'pull', '--ff-only'])

os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
if str(REPO / 'severstral-osr' / 'src') not in sys.path:
    sys.path.insert(0, str(REPO / 'severstral-osr' / 'src'))

print('cwd:', Path.cwd())

In [ ]:
# Mount drive and generate colab configs
import yaml
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

sev = Path('/content/drive/MyDrive/datasets/severstal')
assert sev.exists(), f'Missing {sev}'
assert (sev / 'train.csv').exists(), 'Missing train.csv'
assert (sev / 'train_images').exists(), 'Missing train_images'

base = yaml.safe_load(Path('severstral-osr/configs/default.yaml').read_text())
base['device'] = 'cuda'
base['severstal']['data_root'] = str(sev)
base['output_dir'] = '/content/drive/MyDrive/fyp_outputs/severstral_osr'
base['export_dir'] = '/content/drive/MyDrive/fyp_outputs/severstral_osr/exports'

Path('severstral-osr/configs/default.colab.yaml').write_text(yaml.safe_dump(base, sort_keys=False))
print('wrote severstral-osr/configs/default.colab.yaml')

for s in ['a', 'b', 'c', 'd']:
    split_cfg = yaml.safe_load(Path(f'severstral-osr/configs/split_{s}.yaml').read_text())
    merged = yaml.safe_load(yaml.safe_dump(base))
    merged.update(split_cfg)
    out = Path(f'severstral-osr/configs/split_{s}.colab.yaml')
    out.write_text(yaml.safe_dump(merged, sort_keys=False))
    print('wrote', out)

In [ ]:
# Export Severstal defect stats and single-label lists
from export_single_label_lists import main as export_lists

export_lists('severstral-osr/configs/default.colab.yaml')

In [ ]:
# Stage-1 PatchCore (run once)
from notebook_entrypoints import run_stage1

run_stage1('severstral-osr/configs/default.colab.yaml')

In [ ]:
# Run split pipelines (safe to rerun; skip_if_complete=True)
import json, time
from pathlib import Path
from notebook_entrypoints import run_split_pipeline

cfgs = [
    'severstral-osr/configs/split_a.colab.yaml',
    'severstral-osr/configs/split_b.colab.yaml',
    'severstral-osr/configs/split_c.colab.yaml',
    'severstral-osr/configs/split_d.colab.yaml',
]

out_root = Path('/content/drive/MyDrive/fyp_outputs/severstral_osr')
for c in cfgs:
    split = Path(c).stem.replace('.colab', '')
    m = out_root / split / 'cascade' / 'metrics.json'
    if m.exists():
        print('[skip]', split, 'already has cascade metrics')
        continue
    t = time.time()
    run_split_pipeline(c, skip_if_complete=True)
    print('[done]', split, f'{time.time()-t:.1f}s')

In [ ]:
# Aggregate and print summary
import json
import pandas as pd
import yaml
from pathlib import Path

out_root = Path('/content/drive/MyDrive/fyp_outputs/severstral_osr')
rows = []
for s in ['split_a', 'split_b', 'split_c', 'split_d']:
    p_osr = out_root / s / 'osr' / 'metrics.json'
    p_cas = out_root / s / 'cascade' / 'metrics.json'
    if not p_osr.exists() or not p_cas.exists():
        print('missing metrics for', s)
        continue
    osr = json.loads(p_osr.read_text())
    cas = json.loads(p_cas.read_text())
    split_cfg_path = Path(f'severstral-osr/configs/{s}.yaml')
    known_classes = ''
    if split_cfg_path.exists():
        known_classes = ','.join(yaml.safe_load(split_cfg_path.read_text())['known_classes'])
    rows.append({
        'split': s,
        'known_classes': known_classes,
        'auroc_known_unknown': osr.get('auroc_known_unknown'),
        'tpr_unknown': osr.get('tpr_unknown'),
        'fpr_known': osr.get('fpr_known'),
        'open_set_acc': osr.get('open_set_acc'),
        'known_accuracy': osr.get('known_accuracy'),
        'tau_source': cas.get('patchcore_threshold_source'),
        'tau': cas.get('patchcore_threshold'),
        'kappa': cas.get('kappa'),
        'fusion_rule': cas.get('fusion_rule'),
        'stage1_pass_rate_known': cas.get('stage1_pass_rate_known'),
        'stage1_pass_rate_unknown': cas.get('stage1_pass_rate_unknown'),
        'tpr_unknown_system': cas.get('tpr_unknown_system'),
        'fpr_known_system': cas.get('fpr_known_system'),
    })

df = pd.DataFrame(rows).sort_values('split')
if len(df) > 0:
    display(df)
    display(df.mean(numeric_only=True).to_frame('mean').T)
    out_csv = out_root / 'severstral_osr_summary.csv'
    df.to_csv(out_csv, index=False)
    print('Wrote:', out_csv)
else:
    print('No metrics found yet.')
